# Anomaly Detection Baselines for Credit Card Fraud

This notebook benchmarks **Isolation Forest** and **One-Class SVM** as unsupervised anomaly detection baselines, then compares them against the best supervised model (XGBoost + SMOTE, PR-AUC=0.879, ROC-AUC=0.985, F1=0.89).

Dataset: `creditcard.csv` — 284,807 transactions, 0.17% fraud rate, features V1–V28 + Time + Amount + Class.

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Load dataset
df = pd.read_csv("creditcard.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution (raw):")
print(df['Class'].value_counts())
print(f"\nFraud rate: {df['Class'].mean()*100:.4f}%")

In [ ]:
# Feature engineering: log-transform Amount, drop raw Amount and Time
df['Log_Amount'] = np.log1p(df['Amount'])
df = df.drop(columns=['Amount', 'Time'])

# Separate features and target
X = df.drop(columns=['Class'])
y = df['Class']

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size:  {X_test.shape[0]:,}")
print(f"\nTrain class distribution:")
print(y_train.value_counts())
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")
print(f"\nTest class distribution:")
print(y_test.value_counts())
print(f"Test fraud rate:  {y_test.mean()*100:.4f}%")

In [ ]:
# StandardScaler — fit on train only, transform both train and test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled mean (first feature): {X_train_scaled[:, 0].mean():.6f}")
print(f"X_train_scaled std  (first feature): {X_train_scaled[:, 0].std():.6f}")

## 2. Isolation Forest

Isolation Forest is trained **unsupervised** on `X_train` only — it does not use `y_train`. It isolates observations by randomly selecting a feature and a split value, so anomalies (fraud) require fewer splits to isolate.

- `contamination=0.002` approximates the ~0.17% fraud rate
- Predictions: `-1` = anomaly (fraud), `1` = normal — remapped to `1`/`0`
- `decision_function` scores are negated so that higher score = more anomalous

In [ ]:
print("Training Isolation Forest...")
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.002,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso_forest.fit(X_train_scaled)
print("Isolation Forest training complete.")

In [ ]:
# Predict on test set
# Raw predictions: -1 (anomaly/fraud) or 1 (normal)
iso_raw_preds = iso_forest.predict(X_test_scaled)

# Remap: -1 -> 1 (fraud), 1 -> 0 (normal)
iso_preds = np.where(iso_raw_preds == -1, 1, 0)

# Anomaly scores: negate decision_function so higher = more anomalous
iso_scores = -iso_forest.decision_function(X_test_scaled)

# Compute metrics
iso_pr_auc  = average_precision_score(y_test, iso_scores)
iso_roc_auc = roc_auc_score(y_test, iso_scores)
iso_f1      = f1_score(y_test, iso_preds)
iso_prec    = precision_score(y_test, iso_preds, zero_division=0)
iso_rec     = recall_score(y_test, iso_preds)
iso_cm      = confusion_matrix(y_test, iso_preds)

print("=" * 45)
print("Isolation Forest — Test Set Results")
print("=" * 45)
print(f"PR-AUC    : {iso_pr_auc:.4f}")
print(f"ROC-AUC   : {iso_roc_auc:.4f}")
print(f"F1 Score  : {iso_f1:.4f}")
print(f"Precision : {iso_prec:.4f}")
print(f"Recall    : {iso_rec:.4f}")
print(f"\nConfusion Matrix:")
print(iso_cm)

## 3. One-Class SVM

One-Class SVM learns a boundary around the "normal" training data using a kernel function. Points outside the boundary are classified as anomalies.

- `nu=0.002` is an upper bound on the fraction of training errors (approximates fraud rate)
- **Note:** One-Class SVM is very slow on large datasets. We subsample `X_train` to **10,000 rows** for fitting.
- Same remapping and evaluation as Isolation Forest

In [ ]:
# Subsample X_train to 10,000 rows for One-Class SVM (computational efficiency)
SUBSAMPLE_SIZE = 10_000
rng = np.random.default_rng(RANDOM_STATE)
subsample_idx = rng.choice(len(X_train_scaled), size=SUBSAMPLE_SIZE, replace=False)
X_train_sub = X_train_scaled[subsample_idx]

print(f"One-Class SVM will be trained on {SUBSAMPLE_SIZE:,} rows (subsampled from {len(X_train_scaled):,})")
print("Training One-Class SVM (this may take a minute)...")

oc_svm = OneClassSVM(kernel='rbf', nu=0.002, gamma='scale')
oc_svm.fit(X_train_sub)
print("One-Class SVM training complete.")

In [ ]:
# Predict on test set
# Raw predictions: -1 (anomaly/fraud) or 1 (normal)
svm_raw_preds = oc_svm.predict(X_test_scaled)

# Remap: -1 -> 1 (fraud), 1 -> 0 (normal)
svm_preds = np.where(svm_raw_preds == -1, 1, 0)

# Anomaly scores: negate decision_function so higher = more anomalous
svm_scores = -oc_svm.decision_function(X_test_scaled)

# Compute metrics
svm_pr_auc  = average_precision_score(y_test, svm_scores)
svm_roc_auc = roc_auc_score(y_test, svm_scores)
svm_f1      = f1_score(y_test, svm_preds)
svm_prec    = precision_score(y_test, svm_preds, zero_division=0)
svm_rec     = recall_score(y_test, svm_preds)
svm_cm      = confusion_matrix(y_test, svm_preds)

print("=" * 45)
print("One-Class SVM — Test Set Results")
print("=" * 45)
print(f"PR-AUC    : {svm_pr_auc:.4f}")
print(f"ROC-AUC   : {svm_roc_auc:.4f}")
print(f"F1 Score  : {svm_f1:.4f}")
print(f"Precision : {svm_prec:.4f}")
print(f"Recall    : {svm_rec:.4f}")
print(f"\nConfusion Matrix:")
print(svm_cm)

## 4. Comparison Table

Comparing both unsupervised anomaly detectors against the supervised baselines from our prior experiments.

In [ ]:
# Build comparison DataFrame
comparison_data = {
    'Model': [
        'Isolation Forest (unsupervised)',
        'One-Class SVM (unsupervised)',
        'LR + SMOTE (supervised)',
        'RF + Class Weight (supervised)',
        'XGBoost + SMOTE (supervised) *BEST*'
    ],
    'PR-AUC': [
        round(iso_pr_auc, 3),
        round(svm_pr_auc, 3),
        0.800,
        0.887,
        0.879
    ],
    'ROC-AUC': [
        round(iso_roc_auc, 3),
        round(svm_roc_auc, 3),
        0.977,
        0.984,
        0.985
    ],
    'F1': [
        round(iso_f1, 3),
        round(svm_f1, 3),
        0.14,
        0.87,
        0.89
    ],
    'Type': [
        'Unsupervised',
        'Unsupervised',
        'Supervised',
        'Supervised',
        'Supervised'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')

print("=" * 80)
print("Model Comparison: Unsupervised Anomaly Detection vs Supervised Classifiers")
print("=" * 80)
print(comparison_df.to_string())
print("\n* Best supervised model highlighted")

## 5. ROC and PR Curves — All Models

In [ ]:
# Ensure assets directory exists
os.makedirs("assets", exist_ok=True)

# Compute curves for Isolation Forest
iso_fpr, iso_tpr, _ = roc_curve(y_test, iso_scores)
iso_precision_curve, iso_recall_curve, _ = precision_recall_curve(y_test, iso_scores)

# Compute curves for One-Class SVM
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_scores)
svm_precision_curve, svm_recall_curve, _ = precision_recall_curve(y_test, svm_scores)

# Supervised model reference values
supervised_models = {
    'LR + SMOTE':          {'roc_auc': 0.977, 'pr_auc': 0.800, 'color': '#2ca02c', 'ls': '--'},
    'RF + Class Weight':   {'roc_auc': 0.984, 'pr_auc': 0.887, 'color': '#ff7f0e', 'ls': '-.'},
    'XGBoost + SMOTE':     {'roc_auc': 0.985, 'pr_auc': 0.879, 'color': '#d62728', 'ls': ':'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Anomaly Detection Baselines vs Supervised Models\n'
    'Credit Card Fraud Detection',
    fontsize=14, fontweight='bold', y=1.01
)

# ── ROC Curve (left) ──────────────────────────────────────────────────────────
ax_roc = axes[0]

ax_roc.plot(iso_fpr, iso_tpr, color='#1f77b4', lw=2,
            label=f'Isolation Forest (AUC={iso_roc_auc:.3f})')
ax_roc.plot(svm_fpr, svm_tpr, color='#9467bd', lw=2,
            label=f'One-Class SVM (AUC={svm_roc_auc:.3f})')

# Supervised reference lines (horizontal annotations)
for name, vals in supervised_models.items():
    ax_roc.axhline(
        y=vals['roc_auc'], color=vals['color'],
        linestyle=vals['ls'], lw=1.5,
        label=f"{name} (AUC={vals['roc_auc']:.3f})"
    )

ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random (AUC=0.500)')
ax_roc.set_xlabel('False Positive Rate', fontsize=12)
ax_roc.set_ylabel('True Positive Rate', fontsize=12)
ax_roc.set_title('ROC Curve', fontsize=13)
ax_roc.legend(fontsize=8, loc='lower right')
ax_roc.set_xlim([0.0, 1.0])
ax_roc.set_ylim([0.0, 1.05])
ax_roc.grid(True, alpha=0.3)

# ── PR Curve (right) ──────────────────────────────────────────────────────────
ax_pr = axes[1]

ax_pr.plot(iso_recall_curve, iso_precision_curve, color='#1f77b4', lw=2,
           label=f'Isolation Forest (AP={iso_pr_auc:.3f})')
ax_pr.plot(svm_recall_curve, svm_precision_curve, color='#9467bd', lw=2,
           label=f'One-Class SVM (AP={svm_pr_auc:.3f})')

# Supervised reference lines (horizontal annotations)
for name, vals in supervised_models.items():
    ax_pr.axhline(
        y=vals['pr_auc'], color=vals['color'],
        linestyle=vals['ls'], lw=1.5,
        label=f"{name} (AP={vals['pr_auc']:.3f})"
    )

# Baseline: random classifier precision = fraud rate
fraud_rate = y_test.mean()
ax_pr.axhline(y=fraud_rate, color='k', linestyle='--', lw=1, alpha=0.4,
              label=f'Random (AP={fraud_rate:.4f})')

ax_pr.set_xlabel('Recall', fontsize=12)
ax_pr.set_ylabel('Precision', fontsize=12)
ax_pr.set_title('Precision-Recall Curve', fontsize=13)
ax_pr.legend(fontsize=8, loc='upper right')
ax_pr.set_xlim([0.0, 1.0])
ax_pr.set_ylim([0.0, 1.05])
ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
save_path = "assets/AnomalyDetection-curves.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {save_path}")

## 6. Discussion

### Why Unsupervised Methods Underperform Supervised Models

Unsupervised anomaly detectors — Isolation Forest and One-Class SVM — are trained without access to fraud labels. Their decision boundaries are shaped entirely by the *structure* of the feature space, not by what distinguishes fraud from legitimate transactions. Since credit card fraud is a carefully crafted deception, many fraudulent transactions are engineered to look like ordinary purchases, making purely structure-based separation unreliable. Supervised models, by contrast, learn the precise feature combinations that correlate with fraud, yielding far sharper boundaries and higher precision.

### When Anomaly Detection Is Useful

Unsupervised methods remain valuable in scenarios where labeled fraud data is **scarce or unavailable**:

- **Cold-start problems**: A new payment system with no fraud history yet.
- **Novel fraud patterns**: Emerging attack vectors that existing labels do not cover.
- **Pre-labeling triage**: Flagging candidates for human review before labels are collected.
- **Complementary signal**: Anomaly scores can be added as features to a supervised model to capture distributional outliers the classifier might otherwise miss.

### The Precision-Recall Tradeoff

Anomaly detectors tend to achieve **high recall** — they cast a wide net and catch many true frauds — but at the cost of **very low precision**. A large fraction of legitimate transactions are flagged, leading to excessive false positives. For a real-world fraud operations team, low-precision alerts are expensive: each alert requires analyst time, and unnecessary declines frustrate customers. This trade-off is clearly visible in the PR curves above, where both unsupervised models sit far below the supervised baselines in the precision dimension.

### Isolation Forest vs One-Class SVM at Scale

At the scale of this dataset (284,807 transactions), **Isolation Forest is strongly preferred** over One-Class SVM:

| Property | Isolation Forest | One-Class SVM |
|---|---|---|
| Training complexity | O(n log n) | O(n²) to O(n³) |
| Scalability | Excellent — full dataset | Poor — requires subsampling |
| Parallelizable | Yes (`n_jobs=-1`) | No |
| Performance (PR-AUC) | Comparable or better | Comparable or lower |

One-Class SVM required subsampling to 10,000 rows to run in reasonable time, which may further degrade its generalization. Isolation Forest trained on the full 227,845-row training set without subsampling, making it the practical choice for production-scale anomaly detection.

### Conclusion

For this dataset — where labeled fraud examples exist — **XGBoost + SMOTE (PR-AUC=0.879, ROC-AUC=0.985, F1=0.89)** remains the best model. Unsupervised baselines serve as useful diagnostics and fallbacks, but cannot compete with supervised learning when ground-truth labels are available.